# Figure 5: forward-drug query

This notebook retains the decitabine/leukemia case calculation and result-display blocks used for the Figure 5 query panel. The Tanimoto neighbourhood and benchmark are supplied by the selected accompanying scripts.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path('/Users/dudu/Documents/3_Project/12_PxFquery')
TASK_ROOT = PROJECT_ROOT / '5_phase_translation/G-011_paper_figure_planning/T-149_figure5_forward_drug_query_notebook_figure_workbench'
CASE_ID = 'R15C01_DECITABINE_LEUKEMIA'
CASE_TABLE = TASK_ROOT / '4_artifact/5_table/figure5/panel_a_iter/round15_nucleoside_refine_queries_v0522' / CASE_ID
SELECTED_TABLE = TASK_ROOT / '4_artifact/5_table/figure5/panel_a_selected' / CASE_ID
PIC_DIR = TASK_ROOT / '4_artifact/4_picture/figure5/panel_a_selected' / CASE_ID
PERSIST_DIR = TASK_ROOT / '4_artifact/2_persist'

route_png = PIC_DIR / 'figure5a_r15c01_decitabine_leukemia_route_graph_full_labels_editable_text.png'
route_pdf = PIC_DIR / 'figure5a_r15c01_decitabine_leukemia_route_graph_full_labels_editable_text.pdf'
route_svg = PIC_DIR / 'figure5a_r15c01_decitabine_leukemia_route_graph_full_labels_editable_text.svg'
ranked_csv = CASE_TABLE / 'ranked_results.csv'
route_csv = CASE_TABLE / 'route_summary.csv'
answer_txt = CASE_TABLE / 'answer.txt'

for path in [route_png, route_pdf, route_svg, ranked_csv, route_csv, answer_txt]:
    assert path.exists(), path

print('Route PNG:', route_png)
print('Editable PDF:', route_pdf)
print('Editable SVG:', route_svg)
print('Ranked results:', ranked_csv)
print('Route summary:', route_csv)

In [ ]:
display(Image(filename=str(route_png)))

In [ ]:
route = pd.read_csv(route_csv)
route_display = route[['status','cell','perturbation_alias','perturbation','tier','cell_match_type','perturbation_match_type','route_quality']].copy()
route_display

In [ ]:
ranked = pd.read_csv(ranked_csv)
cols = ['rank','label','direction','score','support_routes','support_cells','cells','route_ids','direction_consistent']
ranked[cols].head(16)

In [ ]:
activated = ranked[(ranked['direction'] == 'activated') & (ranked['support_routes'] >= 2)][['rank','label','score','support_routes','support_cells']].head(8)
suppressed = ranked[(ranked['direction'] == 'suppressed') & (ranked['support_routes'] >= 2)][['rank','label','score','support_routes','support_cells']].head(8)
print('Activated consensus programs')
display(activated)
print('Suppressed consensus programs')
display(suppressed)

In [ ]:
old_answer = answer_txt.read_text(encoding='utf-8').strip()
print(old_answer)

## Evidence network graph\n\nThe original rendering code retained for this figure.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import textwrap
import time
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path("/Users/dudu/Documents/3_Project/12_PxFquery")
TASK_ROOT = PROJECT_ROOT / "5_phase_translation" / "G-011_paper_figure_planning" / "T-149_figure5_forward_drug_query_notebook_figure_workb"
PKG_ROOT = TASK_ROOT / "3_execution" / "t144_package_source_for_figure5a_round5"
SRC_ROOT = PKG_ROOT / "src"

ROUND_ID = "round5_clean_route_graph_v0522"
OUT_PICTURE = TASK_ROOT / "4_artifact" / "4_picture" / "figure5" / "panel_a_iter" / ROUND_ID
OUT_TABLE = TASK_ROOT / "4_artifact" / "5_table" / "figure5" / "panel_a_iter" / ROUND_ID
OUT_TMP = TASK_ROOT / "4_artifact" / "1_tmp" / "panel_a_iter" / ROUND_ID

CASES = [
    {
        "case_id": "R5C01_CLEAN_TAMOXIFEN_BREAST",
        "drug_theme": "tamoxifen-like anti-estrogen treatment in breast cancer",
        "why_test": "Clean disease + drug-family query; tests whether SERM/tamoxifen structural neighbors produce a readable drug-to-function route graph.",
        "query": "In breast cancer models, what functional programs change after tamoxifen-like anti-estrogen treatment?",
    },
    {
        "case_id": "R5C02_CLEAN_GEMCITABINE_PDAC",
        "drug_theme": "gemcitabine-like nucleoside chemotherapy in pancreatic cancer",
        "why_test": "Clean disease + chemotherapy-family query; tests whether nucleoside analog drug neighbors give the clearest bridge to Figure 5 drug-similarity evidence.",
        "query": "In pancreatic cancer models, what functional programs change after gemcitabine-like nucleoside chemotherapy?",
    },
    {
        "case_id": "R5C03_CLEAN_BRAF_MEK_MELANOMA",
        "drug_theme": "BRAF/MEK inhibitor treatment in melanoma",
        "why_test": "Clean targeted-therapy query; tests whether a canonical MAPK drug example gives an intuitive route graph and functional answer.",
        "query": "In melanoma models, what functional programs change after BRAF or MEK inhibitor treatment such as vemurafenib or trametinib?",
    },
    {
        "case_id": "R5C04_CLEAN_EGFR_LUNG",
        "drug_theme": "EGFR inhibitor treatment in lung cancer",
        "why_test": "Clean RTK inhibitor query; tests whether a familiar lung-cancer targeted therapy produces a direct reader-friendly Figure 5A route graph.",
        "query": "In lung cancer models, what functional programs change after EGFR inhibitor treatment such as gefitinib or erlotinib?",
    },
]


def load_env() -> None:
    sys.path.insert(0, str(SRC_ROOT))
    os.environ["PYTHONPATH"] = str(SRC_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
    for env_path in [
        Path.home() / ".env",
        PROJECT_ROOT / ".env",
        PROJECT_ROOT / "4_phase_development" / "G-035_goal_ms8_human_usable_package" / "T-144_ms8_14_user_value_l5_output_rework" / "1_asset" / ".env",
    ]:
        if not env_path.exists():
            continue
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())


def write_table(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)


def build_pick_sheet(summary_df: pd.DataFrame, output_path: Path) -> None:
    from PIL import Image, ImageDraw, ImageFont

    try:
        font_title = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial Bold.ttf", 28)
        font_case = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial Bold.ttf", 17)
        font_small = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 13)
        font_header = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial Bold.ttf", 15)
    except Exception:
        font_title = font_case = font_small = font_header = ImageFont.load_default()

    cols = [
        ("01_evidence_match_map", "Evidence map"),
        ("02_forward_route_graph", "Route graph"),
        ("03_function_match_heatmap", "Heatmap"),
        ("04_function_consensus_bar", "Consensus"),
    ]
    thumb_w = 380
    thumb_h = 300
    label_w = 440
    pad = 20
    header_h = 96
    row_h = thumb_h + 70
    sheet_w = label_w + len(cols) * thumb_w + (len(cols) + 2) * pad
    sheet_h = header_h + len(summary_df) * row_h + pad
    canvas = Image.new("RGB", (sheet_w, sheet_h), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((pad, 18), "Figure 5A clean-query route graph candidates (T144 0.5.22)", fill=(0, 0, 0), font=font_title)
    for j, (_, title) in enumerate(cols):
        x = label_w + pad + j * (thumb_w + pad)
        draw.text((x + 6, 62), title, fill=(0, 0, 0), font=font_header)

    for i, row in summary_df.reset_index(drop=True).iterrows():
        cid = row["case_id"]
        y = header_h + i * row_h
        if i % 2 == 0:
            draw.rectangle((0, y, sheet_w, y + row_h), fill=(248, 248, 248))
        draw.line((0, y, sheet_w, y), fill=(220, 220, 220), width=1)
        draw.text((pad, y + 14), cid, fill=(0, 0, 0), font=font_case)
        text = f"{row['drug_theme']}\n{row['query']}\nRoutes={row['route_rows']} Functions={row['function_rows']} Error={bool(row['error'])}"
        lines: list[str] = []
        for part in text.split("\n"):
            lines.extend(textwrap.wrap(part, width=52))
        for k, line in enumerate(lines[:9]):
            draw.text((pad, y + 42 + k * 17), line, fill=(35, 35, 35), font=font_small)
        for j, (suffix, _) in enumerate(cols):
            matches = sorted((OUT_PICTURE / cid / "png").glob(f"*_{suffix}.png"))
            if not matches:
                continue
            img = Image.open(matches[0]).convert("RGB")
            img.thumbnail((thumb_w, thumb_h), Image.Resampling.LANCZOS)
            x = label_w + pad + j * (thumb_w + pad) + (thumb_w - img.width) // 2
            yy = y + 48 + (thumb_h - img.height) // 2
            canvas.paste(img, (x, yy))
            draw.rectangle((x, yy, x + img.width, yy + img.height), outline=(210, 210, 210), width=1)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output_path, quality=95)


def score_row(row: dict) -> dict:
    pert_types = str(row.get("perturbation_match_types") or "")
    aliases = [x for x in str(row.get("perturbation_aliases") or "").split("|") if x]
    route_rows = int(row.get("route_rows") or 0)
    function_rows = int(row.get("function_rows") or 0)
    route_graph_score = min(5, 1 + route_rows // 3)
    if "structural_neighbor" in pert_types or "mechanism_representative" in pert_types:
        route_graph_score = min(5, route_graph_score + 1)
    if len(aliases) >= 2:
        route_graph_score = min(5, route_graph_score + 1)
    heatmap_score = 4 if function_rows >= 64 else 3 if function_rows >= 32 else 1
    answer_score = 4 if function_rows >= 64 else 3 if function_rows >= 32 else 1
    return {
        "route_graph_score": route_graph_score,
        "heatmap_score": heatmap_score,
        "answer_score": answer_score,
        "overall_score": round((route_graph_score + heatmap_score + answer_score) / 3, 2),
    }


def main() -> None:
    load_env()
    from pxfquery import PxFQuery
    import pxfquery

    OUT_PICTURE.mkdir(parents=True, exist_ok=True)
    OUT_TABLE.mkdir(parents=True, exist_ok=True)
    OUT_TMP.mkdir(parents=True, exist_ok=True)
    manifest = {
        "round_id": ROUND_ID,
        "package_root": str(PKG_ROOT),
        "pxfquery_file": str(Path(pxfquery.__file__).resolve()),
        "version": pxfquery.__version__,
        "cases": CASES,
        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    (OUT_TMP / "run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    summary_rows = []
    for case in CASES:
        cid = case["case_id"]
        case_picture = OUT_PICTURE / cid
        case_table = OUT_TABLE / cid
        case_tmp = OUT_TMP / cid
        case_picture.mkdir(parents=True, exist_ok=True)
        case_table.mkdir(parents=True, exist_ok=True)
        case_tmp.mkdir(parents=True, exist_ok=True)
        started = time.perf_counter()
        route_rows: list[dict] = []
        function_rows: list[dict] = []
        ranked_rows: list[dict] = []
        error = ""
        try:
            client = PxFQuery()
            qdata = client.tl.parse(case["query"], top_n=8, progress=False)
            client.tl.answer(qdata, progress=False)
            answer = client.get.answer(qdata)
            route_rows = list(answer.tables.get("route_summary", []))
            function_rows = list(answer.tables.get("route_function_results", []))
            ranked_rows = list(answer.tables.get("ranked_results", []))
            write_table(route_rows, case_table / "route_summary.csv")
            write_table(function_rows, case_table / "route_function_results.csv")
            write_table(ranked_rows, case_table / "ranked_results.csv")
            (case_table / "answer.txt").write_text(str(answer), encoding="utf-8")
            for fmt in ["png", "pdf"]:
                client.tl.figures(qdata, output_dir=case_picture / fmt, prefix=cid.lower(), format=fmt)
        except Exception as exc:
            error = repr(exc)
            (case_tmp / "error.txt").write_text(error, encoding="utf-8")
        elapsed = time.perf_counter() - started
        route_df = pd.DataFrame(route_rows)
        fn_df = pd.DataFrame(function_rows)
        row = {
            **case,
            "elapsed_seconds": round(elapsed, 3),
            "error": error,
            "route_rows": len(route_rows),
            "function_rows": len(function_rows),
            "ranked_rows": len(ranked_rows),
            "unique_cells": "|".join(sorted(route_df["cell"].dropna().astype(str).unique())) if "cell" in route_df else "",
            "unique_perturbations": "|".join(sorted(route_df["perturbation"].dropna().astype(str).unique())) if "perturbation" in route_df else "",
            "perturbation_aliases": "|".join(sorted(route_df["perturbation_alias"].dropna().astype(str).unique())) if "perturbation_alias" in route_df else "",
            "cell_match_types": "|".join(sorted(route_df["cell_match_type"].dropna().astype(str).unique())) if "cell_match_type" in route_df else "",
            "perturbation_match_types": "|".join(sorted(route_df["perturbation_match_type"].dropna().astype(str).unique())) if "perturbation_match_type" in route_df else "",
            "top_activated": "|".join(fn_df.loc[fn_df.get("direction", "") == "activated", "label"].head(5).astype(str)) if "direction" in fn_df and "label" in fn_df else "",
            "top_suppressed": "|".join(fn_df.loc[fn_df.get("direction", "") == "suppressed", "label"].head(5).astype(str)) if "direction" in fn_df and "label" in fn_df else "",
        }
        row.update(score_row(row))
        summary_rows.append(row)
        print(f"{cid}: routes={len(route_rows)} functions={len(function_rows)} ranked={len(ranked_rows)} error={bool(error)} elapsed={elapsed:.1f}s", flush=True)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(OUT_TABLE / f"{ROUND_ID}_summary.csv", index=False)
    build_pick_sheet(summary_df, OUT_PICTURE / f"{ROUND_ID}_pick_sheet.png")
    (OUT_TMP / "run_complete.json").write_text(
        json.dumps(
            {
                "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                "summary_path": str(OUT_TABLE / f"{ROUND_ID}_summary.csv"),
                "picture_dir": str(OUT_PICTURE),
            },
            indent=2,
        ),
        encoding="utf-8",
    )


if __name__ == "__main__":
    main()


In [ ]:
from __future__ import annotations

import math
import textwrap
from pathlib import Path

import pandas as pd


TASK_ROOT = Path(__file__).resolve().parents[1]
CASE_ID = "R15C01_DECITABINE_LEUKEMIA"
ROUND_ID = "round15_nucleoside_refine_queries_v0522"

TABLE_DIR = (
    TASK_ROOT
    / "4_artifact"
    / "5_table"
    / "figure5"
    / "panel_a_iter"
    / ROUND_ID
    / CASE_ID
)
OUT_DIR = (
    TASK_ROOT
    / "4_artifact"
    / "4_picture"
    / "figure5"
    / "panel_a_selected"
    / CASE_ID
)

PERT_NAME = {
    "BRD-K79254416": "decitabine",
    "BRD-K33106058": "cytarabine",
    "BRD-A18929998": "cytarabine",
    "BRD-K15108141": "gemcitabine",
}

DISPLAY_FUNCTIONS = [
    "MP17 Interferon/MHC-II (I)",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
    "MP5 Stress ",
    "MP20 MYC",
    "HALLMARK_MYC_TARGETS_V1",
    "MP11 Translation initiation",
]


def _display_function(label: str) -> str:
    text = str(label or "").strip()
    text = text.replace("HALLMARK_", "").replace("_", " ")
    return text


def _wrap(text: str, width: int) -> str:
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False, break_on_hyphens=False))


def _layer_positions(items: list[str], y: float, x0: float, x1: float) -> dict[str, tuple[float, float]]:
    if len(items) == 1:
        return {items[0]: ((x0 + x1) / 2, y)}
    step = (x1 - x0) / (len(items) - 1)
    return {item: (x0 + i * step, y) for i, item in enumerate(items)}


def _draw_node(ax, xy, label, face, edge, *, size, fs, width, yoff):
    x, y = xy
    ax.scatter([x], [y], s=size, facecolor=face, edgecolor=edge, linewidth=1.6, zorder=4)
    ax.text(
        x,
        y - yoff,
        _wrap(label, width),
        ha="center",
        va="top",
        fontsize=fs,
        linespacing=1.04,
        zorder=5,
        color="#111827",
    )


def main() -> None:
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    plt.rcParams["svg.fonttype"] = "none"
    plt.rcParams["font.family"] = "Arial"
    plt.rcParams["text.usetex"] = False

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    route = pd.read_csv(TABLE_DIR / "route_summary.csv")
    funcs = pd.read_csv(TABLE_DIR / "route_function_results.csv")

    executed = route[route["status"] == "executed"].copy()
    executed["drug"] = executed.apply(
        lambda row: PERT_NAME.get(str(row["perturbation"]), str(row["perturbation_alias"] or row["perturbation"])),
        axis=1,
    )
    executed = executed[["cell", "drug", "perturbation_match_type"]].drop_duplicates()

    funcs["drug"] = funcs.apply(
        lambda row: PERT_NAME.get(str(row["perturbation"]), str(row["perturbation_alias"] or row["perturbation"])),
        axis=1,
    )
    sub = funcs[funcs["label"].isin(DISPLAY_FUNCTIONS)].copy()

    cells = ["JURKAT", "THP1", "SKM1", "NOMO1"]
    drugs = ["decitabine", "cytarabine", "gemcitabine"]
    functions = DISPLAY_FUNCTIONS

    cell_pos = {f"cell:{k}": v for k, v in _layer_positions(cells, 0.78, 0.13, 0.94).items()}
    drug_pos = {f"drug:{k}": v for k, v in _layer_positions(drugs, 0.49, 0.13, 0.94).items()}
    fn_pos = {f"fn:{k}": v for k, v in _layer_positions(functions, 0.17, 0.08, 0.98).items()}
    pos = {**cell_pos, **drug_pos, **fn_pos}

    fig, ax = plt.subplots(figsize=(11.8, 7.1))
    ax.set_axis_off()
    ax.set_xlim(-0.08, 1.03)
    ax.set_ylim(0.00, 0.98)

    ax.text(-0.055, 0.78, "Cell\nContext", ha="left", va="center", fontsize=10, weight="bold", color="#374151")
    ax.text(-0.055, 0.49, "Perturbation\nEvidence", ha="left", va="center", fontsize=10, weight="bold", color="#374151")
    ax.text(-0.055, 0.17, "Consensus\nPrograms", ha="left", va="center", fontsize=10, weight="bold", color="#374151")

    for _, row in executed.iterrows():
        c = f"cell:{row['cell']}"
        d = f"drug:{row['drug']}"
        if c not in pos or d not in pos:
            continue
        linestyle = "-" if "user_specified" in str(row["perturbation_match_type"]) else "--"
        ax.annotate(
            "",
            xy=pos[d],
            xytext=pos[c],
            arrowprops=dict(arrowstyle="-", color="#9ca3af", lw=1.3, linestyle=linestyle, alpha=0.62),
            zorder=1,
        )

    collapsed: dict[tuple[str, str], list[float]] = {}
    for _, row in sub.iterrows():
        collapsed.setdefault((str(row["drug"]), str(row["label"])), []).append(float(row["score"]))
    max_abs = max([abs(v) for vals in collapsed.values() for v in vals] + [1.0])
    for (drug, fn), vals in collapsed.items():
        d = f"drug:{drug}"
        f = f"fn:{fn}"
        if d not in pos or f not in pos:
            continue
        score = sum(vals) / len(vals)
        color = "#b23a48" if score > 0 else "#33658a"
        lw = 0.65 + 2.8 * min(abs(score) / max_abs, 1.0)
        ax.annotate(
            "",
            xy=pos[f],
            xytext=pos[d],
            arrowprops=dict(arrowstyle="-", color=color, lw=lw, alpha=0.52),
            zorder=2,
        )

    for cell in cells:
        _draw_node(ax, pos[f"cell:{cell}"], cell, "#dbeafe", "#2563eb", size=360, fs=9.0, width=12, yoff=0.050)
    for drug in drugs:
        _draw_node(ax, pos[f"drug:{drug}"], drug, "#fef3c7", "#d97706", size=390, fs=9.0, width=14, yoff=0.050)
    for fn in functions:
        scores = sub[sub["label"] == fn]["score"].astype(float).tolist()
        mean_score = sum(scores) / len(scores) if scores else 0.0
        edge = "#b23a48" if mean_score >= 0 else "#33658a"
        _draw_node(
            ax,
            pos[f"fn:{fn}"],
            _display_function(fn),
            "#f9fafb",
            edge,
            size=300,
            fs=8.2,
            width=18,
            yoff=0.040,
        )

    handles = [
        Line2D([0], [0], color="#9ca3af", lw=1.4, label="cell-to-perturbation match"),
        Line2D([0], [0], color="#b23a48", lw=2.8, label="activated function edge"),
        Line2D([0], [0], color="#33658a", lw=2.8, label="suppressed function edge"),
    ]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.035), ncol=3, frameon=False, fontsize=9)
    ax.set_title("Evidence Match Network", fontsize=15, weight="bold", pad=8)
    fig.tight_layout(rect=(0, 0.03, 1, 0.96))

    pdf = OUT_DIR / "figure5a_r15c01_decitabine_leukemia_route_graph_full_labels_editable_text.pdf"
    png = OUT_DIR / "figure5a_r15c01_decitabine_leukemia_route_graph_full_labels_editable_text.png"
    svg = OUT_DIR / "figure5a_r15c01_decitabine_leukemia_route_graph_full_labels_editable_text.svg"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=180, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.close(fig)
    print(pdf)
    print(png)
    print(svg)


if __name__ == "__main__":
    main()


## 100-query forward-drug benchmark

The forward-drug query stream, evidence-resolution selection, and query-level aggregation used for this panel are retained here. The exact 100 selected IDs and resulting query medians are in `benchmark_data/`.


In [ ]:
from pathlib import Path
import pandas as pd

benchmark_dir = Path("benchmark_data")
query_set = pd.read_csv(benchmark_dir / "query_set.csv")
query_ids = pd.read_csv(benchmark_dir / "query_ids.csv")
query_median = pd.read_csv(benchmark_dir / "query_median.csv")

query_set.head(), query_ids.shape, query_median.groupby(["field", "system"]).query_id.nunique()



In [ ]:
from run_forward_drug_benchmark import (
    build_query_medians,
    first100_canonical_ids,
    question,
    select_candidate_rows,
)

# The production selection rule used the compound query-space metrics, then retained
# the first 100 evidence-resolved queries with nonempty ranked evidence tables.
# The resulting IDs and query-level medians loaded above are the panel inputs.
